# NuminaMath-1.5 retrieval embeddings

Embeds every `problem` of **AI-MO/NuminaMath-1.5** (train split, all rows, dataset order) with the pipeline's
retrieval model **math-similarity/Bert-MLM_arXiv-MP-class_zbMath**, the same way the current
`embedding_NuminaMath_with_Bert-MLM_arXiv-MP-class_zbMath.npy` was made for NuminaMath-CoT:
`SentenceTransformer.encode`, mean pooling, max 512 tokens, float32, **not** normalised (retrieval divides by the norms).

Row `i` of the `.npy` is row `i` of the dataset at the pinned revision written in the manifest.

**Before running**
1. *Settings → Accelerator → GPU T4 x2* (one GPU also works, about twice as slow), *Internet on*.
2. *Add-ons → Secrets*: add `HF_TOKEN` (a Hugging Face token with **write** access) and tick it for this notebook.
3. *Save Version → Save & Run All (Commit)*. It runs in the background; about 1-2 hours on two T4s.

**Uploads to** `mostafabehroozi/numinamath-2025` (dataset repo), folder `embeddings/`:
* `embedding_NuminaMath-1.5_with_Bert-MLM_arXiv-MP-class_zbMath.npy`, float32 `[896215, 768]`, about 2.75 GB
* `NuminaMath-1.5_rows.parquet`, every column except problem/solution, plus `idx` and `problem_sha1` (for filtering and alignment checks)
* `NuminaMath-1.5_embedding_manifest.json`, the recipe, revisions, checks and timings

In [ ]:
# Settings
from pathlib import Path

SOURCE_DATASET = "AI-MO/NuminaMath-1.5"
SOURCE_SPLIT = "train"
TEXT_COLUMN = "problem"
MODEL_ID = "math-similarity/Bert-MLM_arXiv-MP-class_zbMath"

TARGET_REPO = "mostafabehroozi/numinamath-2025"   # dataset repo that receives the files
TARGET_DIR_IN_REPO = "embeddings"
EMBEDDING_FILE = "embedding_NuminaMath-1.5_with_Bert-MLM_arXiv-MP-class_zbMath.npy"
ROWS_FILE = "NuminaMath-1.5_rows.parquet"
MANIFEST_FILE = "NuminaMath-1.5_embedding_manifest.json"

# Recipe check: embed the first rows of NuminaMath-CoT and compare with the existing CoT embedding file, so the new
# file is provably made the same way. The old file is looked up in this repo (None = <your HF user>/<name>).
RECIPE_CHECK = True
OLD_EMBEDDING_REPO = None
OLD_EMBEDDING_REPO_NAME = "analogical-math-rag-results"
OLD_EMBEDDING_FILE = "embedding_NuminaMath_with_Bert-MLM_arXiv-MP-class_zbMath.npy"
RECIPE_CHECK_ROWS = 2000

CHUNK_ROWS = 50_000      # progress is saved after every chunk
BATCH_SIZE = 128         # per GPU
WORK_DIR = Path("/kaggle/working/numina15_embed")
UPLOAD_DIR = WORK_DIR / "upload"  # only the three deliverables live here
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Hugging Face token (Kaggle secret HF_TOKEN) and an early write test, so a bad token fails now, not after the GPU work
import json, os, time, datetime
from huggingface_hub import HfApi

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        raise RuntimeError("Add a Kaggle secret named HF_TOKEN (Add-ons -> Secrets) and tick it for this notebook.") from exc
os.environ["HF_TOKEN"] = HF_TOKEN

api = HfApi(token=HF_TOKEN)
HF_USER = api.whoami()["name"]
print("Hugging Face user:", HF_USER)

try:
    api.repo_info(TARGET_REPO, repo_type="dataset")
    print("Target repo exists:", TARGET_REPO)
except Exception:
    api.create_repo(TARGET_REPO, repo_type="dataset", private=True, exist_ok=True)
    print("Created private dataset repo:", TARGET_REPO)

STARTED_AT = datetime.datetime.utcnow().isoformat() + "Z"
status = {"status": "running", "started_at": STARTED_AT, "source_dataset": SOURCE_DATASET, "model_id": MODEL_ID}
(WORK_DIR / "status.json").write_text(json.dumps(status, indent=2))
api.upload_file(path_or_fileobj=str(WORK_DIR / "status.json"),
                path_in_repo=f"{TARGET_DIR_IN_REPO}/{MANIFEST_FILE}",
                repo_id=TARGET_REPO, repo_type="dataset",
                commit_message="NuminaMath-1.5 embeddings: run started (write test)")
print("Write access confirmed.")

SOURCE_REVISION = api.dataset_info(SOURCE_DATASET).sha
MODEL_REVISION = api.model_info(MODEL_ID).sha
print("Pinned dataset revision:", SOURCE_REVISION)
print("Model revision:", MODEL_REVISION)

In [ ]:
# Load NuminaMath-1.5 at the pinned revision; keep the metadata (every column except the long texts)
import hashlib
import numpy as np
import pandas as pd
from datasets import load_dataset

t0 = time.time()
ds = load_dataset(SOURCE_DATASET, split=SOURCE_SPLIT, revision=SOURCE_REVISION)
print(ds)
print(f"loaded in {time.time() - t0:.0f}s")

raw = ds[TEXT_COLUMN]
texts = [t if isinstance(t, str) else "" for t in raw]
EMPTY_ROWS = int(sum(1 for t in raw if not isinstance(t, str) or not t.strip()))
del raw
N_ROWS = len(texts)
print("rows:", N_ROWS, "| empty problems:", EMPTY_ROWS)

meta = ds.remove_columns([c for c in ("problem", "solution", "messages") if c in ds.column_names]).to_pandas()
meta.insert(0, "idx", np.arange(N_ROWS, dtype=np.int64))
meta["problem_sha1"] = [hashlib.sha1(t.encode("utf-8")).hexdigest() for t in texts]
meta.to_parquet(UPLOAD_DIR / ROWS_FILE, index=False)
print(meta.head())
for col in ("source", "synthetic", "question_type", "problem_is_valid", "solution_is_valid"):
    if col in meta.columns:
        print(f"\n{col}:\n{meta[col].value_counts(dropna=False).to_string()}")
if {"source", "synthetic"} <= set(meta.columns):
    print("\nsynthetic flag by source:\n", pd.crosstab(meta["source"], meta["synthetic"]).to_string())

In [ ]:
# Embedding model on every GPU (one SentenceTransformer per device, encoded from two threads)
import torch
import sentence_transformers
from sentence_transformers import SentenceTransformer

DEVICES = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
print("devices:", DEVICES, "| sentence-transformers", sentence_transformers.__version__, "| torch", torch.__version__)
MODELS = [SentenceTransformer(MODEL_ID, device=d, revision=MODEL_REVISION) for d in DEVICES]
for m in MODELS:
    m.eval()
DIM = MODELS[0].get_sentence_embedding_dimension()
MAX_SEQ = MODELS[0].max_seq_length
print("dim:", DIM, "| max_seq_length:", MAX_SEQ)
print(MODELS[0])

from concurrent.futures import ThreadPoolExecutor
POOL = ThreadPoolExecutor(len(MODELS))


def encode(batch_texts):
    """float32 [n, DIM], same as SentenceTransformer.encode with defaults (no normalisation)."""
    batch_texts = list(batch_texts)
    if not batch_texts:
        return np.zeros((0, DIM), np.float32)
    parts = np.array_split(np.arange(len(batch_texts)), len(MODELS))
    jobs = [POOL.submit(m.encode, [batch_texts[i] for i in p], batch_size=BATCH_SIZE, convert_to_numpy=True,
                        normalize_embeddings=False, show_progress_bar=False)
            for m, p in zip(MODELS, parts) if len(p)]
    return np.concatenate([j.result() for j in jobs]).astype(np.float32, copy=False)


def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)

In [ ]:
# Recipe check against the existing NuminaMath-CoT file (stops the run if the recipe does not match)
class RecipeMismatch(RuntimeError):
    pass


RECIPE_RESULT = {"ran": False}
if RECIPE_CHECK:
    try:
        from huggingface_hub import hf_hub_download
        old_repo = OLD_EMBEDDING_REPO or f"{HF_USER}/{OLD_EMBEDDING_REPO_NAME}"
        hits = [f for f in api.list_repo_files(old_repo, repo_type="dataset") if f.endswith(OLD_EMBEDDING_FILE)]
        if not hits:
            raise FileNotFoundError(f"{OLD_EMBEDDING_FILE} not found in {old_repo}")
        old_path = hf_hub_download(old_repo, hits[0], repo_type="dataset", token=HF_TOKEN)
        old = np.load(old_path, mmap_mode="r")
        cot_files = sorted(f for f in api.list_repo_files("AI-MO/NuminaMath-CoT", repo_type="dataset")
                           if f.startswith("data/train") and f.endswith(".parquet"))
        first = pd.read_parquet(hf_hub_download("AI-MO/NuminaMath-CoT", cot_files[0], repo_type="dataset"),
                                columns=["problem"])
        k = min(RECIPE_CHECK_ROWS, len(first))
        new = encode(first["problem"].tolist()[:k])
        cos = (unit(new) * unit(np.asarray(old[:k], np.float32))).sum(1)
        norm_ratio = np.linalg.norm(new, axis=1) / np.maximum(np.linalg.norm(np.asarray(old[:k]), axis=1), 1e-12)
        RECIPE_RESULT = {"ran": True, "old_file": f"{old_repo}/{hits[0]}", "old_shape": list(old.shape),
                         "old_dtype": str(old.dtype), "rows_compared": int(k),
                         "cosine_mean": float(cos.mean()), "cosine_min": float(cos.min()),
                         "share_cosine_ge_0_9999": float((cos >= 0.9999).mean()),
                         "norm_ratio_mean": float(norm_ratio.mean())}
        print(json.dumps(RECIPE_RESULT, indent=2))
        del old
        if RECIPE_RESULT["cosine_mean"] < 0.999:
            raise RecipeMismatch("Recipe check FAILED: the new embeddings do not match the existing CoT file. Stopping.")
        print("Recipe check passed: same model and recipe as the existing file.")
    except RecipeMismatch:
        raise
    except Exception as exc:
        RECIPE_RESULT = {"ran": False, "skipped_because": repr(exc)}
        print("Recipe check skipped:", repr(exc))

In [ ]:
# Embed every problem into a memory-mapped .npy, chunk by chunk (resumes inside the same session)
from numpy.lib.format import open_memmap

EMB_PATH = UPLOAD_DIR / EMBEDDING_FILE
STATE_PATH = WORK_DIR / "progress.json"
n_chunks = (N_ROWS + CHUNK_ROWS - 1) // CHUNK_ROWS
state = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {}
if EMB_PATH.exists() and state.get("rows") == N_ROWS and state.get("revision") == SOURCE_REVISION:
    emb = open_memmap(EMB_PATH, mode="r+")
    done = set(state["done"])
    print(f"resuming: {len(done)}/{n_chunks} chunks already done")
else:
    emb = open_memmap(EMB_PATH, mode="w+", dtype=np.float32, shape=(N_ROWS, DIM))
    done = set()
    state = {"rows": N_ROWS, "revision": SOURCE_REVISION, "done": []}

t0, rows_this_session = time.time(), 0
for c in range(n_chunks):
    if c in done:
        continue
    s, e = c * CHUNK_ROWS, min(N_ROWS, (c + 1) * CHUNK_ROWS)
    emb[s:e] = encode(texts[s:e])
    emb.flush()
    done.add(c)
    state["done"] = sorted(done)
    STATE_PATH.write_text(json.dumps(state))
    rows_this_session += e - s
    rate = rows_this_session / (time.time() - t0)
    left = sum(min(N_ROWS, (k + 1) * CHUNK_ROWS) - k * CHUNK_ROWS for k in range(n_chunks) if k not in done)
    print(f"chunk {c + 1}/{n_chunks} rows {s}-{e} | {rate:.0f} rows/s | ETA {left / rate / 60:.1f} min", flush=True)
EMBED_SECONDS = time.time() - t0
del emb
print(f"done in {EMBED_SECONDS / 60:.1f} min")

In [ ]:
# Checks: shape, no NaN/zero rows, and a fresh one-by-one re-embedding of random rows matches the stored rows
emb = np.load(EMB_PATH, mmap_mode="r")
assert emb.shape == (N_ROWS, DIM) and emb.dtype == np.float32, (emb.shape, emb.dtype)
bad_nan, zero_rows, norms = 0, 0, []
for s in range(0, N_ROWS, 100_000):
    block = np.asarray(emb[s:s + 100_000])
    bad_nan += int((~np.isfinite(block)).any(1).sum())
    nb = np.linalg.norm(block, axis=1)
    zero_rows += int((nb < 1e-8).sum())
    norms.append(nb)
norms = np.concatenate(norms)
assert bad_nan == 0, f"{bad_nan} rows contain NaN/inf"

rng = np.random.default_rng(0)
sample = np.sort(rng.choice(N_ROWS, size=min(256, N_ROWS), replace=False))
fresh = np.stack([MODELS[0].encode([texts[i]], convert_to_numpy=True, show_progress_bar=False)[0] for i in sample])
spot = (unit(fresh) * unit(np.asarray(emb[sample]))).sum(1)
CHECKS = {"nan_rows": bad_nan, "zero_norm_rows": zero_rows,
          "norm_min": float(norms.min()), "norm_mean": float(norms.mean()), "norm_max": float(norms.max()),
          "spot_check_rows": int(len(sample)), "spot_check_cosine_min": float(spot.min()),
          "spot_check_cosine_mean": float(spot.mean())}
print(json.dumps(CHECKS, indent=2))
assert spot.min() > 0.999, "stored rows do not match a fresh embedding of the same problem"
del emb

In [ ]:
# Manifest, then one commit with the three files
sha = hashlib.sha256()
with open(EMB_PATH, "rb") as fh:
    for blk in iter(lambda: fh.read(1 << 24), b""):
        sha.update(blk)

manifest = {
    "status": "complete", "started_at": STARTED_AT, "finished_at": datetime.datetime.utcnow().isoformat() + "Z",
    "source_dataset": SOURCE_DATASET, "source_revision": SOURCE_REVISION, "split": SOURCE_SPLIT,
    "text_column": TEXT_COLUMN, "rows": N_ROWS, "empty_text_rows": EMPTY_ROWS,
    "row_order": "row i of the .npy is row i of the split at source_revision (see idx/problem_sha1 in the rows file)",
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "dim": DIM, "max_seq_length": MAX_SEQ,
    "pooling": "mean (sentence-transformers Pooling module of the model)", "normalized": False, "dtype": "float32",
    "recipe": "SentenceTransformer(model).encode(problems, convert_to_numpy=True, normalize_embeddings=False)",
    "devices": DEVICES, "batch_size_per_device": BATCH_SIZE, "embed_minutes": round(EMBED_SECONDS / 60, 2),
    "sentence_transformers": sentence_transformers.__version__, "torch": torch.__version__,
    "embedding_sha256": sha.hexdigest(), "embedding_bytes": EMB_PATH.stat().st_size,
    "recipe_check_vs_numinamath_cot_file": RECIPE_RESULT, "checks": CHECKS,
    "files": {"embeddings": EMBEDDING_FILE, "rows": ROWS_FILE},
}
(UPLOAD_DIR / MANIFEST_FILE).write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

t0 = time.time()
api.upload_folder(folder_path=str(UPLOAD_DIR), path_in_repo=TARGET_DIR_IN_REPO, repo_id=TARGET_REPO,
                  repo_type="dataset",
                  commit_message=f"NuminaMath-1.5 problem embeddings ({N_ROWS} x {DIM}, {MODEL_ID}, rev {str(SOURCE_REVISION)[:8]})")
print(f"uploaded in {time.time() - t0:.0f}s")
remote = {f.path: getattr(f, "size", None) for f in api.list_repo_tree(TARGET_REPO, path_in_repo=TARGET_DIR_IN_REPO, repo_type="dataset")}
print(json.dumps(remote, indent=2))
assert remote.get(f"{TARGET_DIR_IN_REPO}/{EMBEDDING_FILE}") == EMB_PATH.stat().st_size, "uploaded size differs"
print(f"\nDONE: https://huggingface.co/datasets/{TARGET_REPO}/tree/main/{TARGET_DIR_IN_REPO}")